# 19 - SimFT: personalized aggregation weighted by function-space similarity (+ a final local fine-tune)

Pre-registered in `docs/2026-10-28-simft-preregistration.md` (read it first; it also lists the evidence behind the design).

**The method.** Each round, after local training, every hospital's model is scored on every hospital's validation
half A (the receiving hospital would do this itself; only the loss numbers matter). Hospital *i*'s next model is a
weighted average of all hospitals' models: it keeps at least 25% of its own, and the rest is a softmax over how much
*better* each neighbour's model is on *its* data (temperature tau). After the last round every hospital fine-tunes its
own model for 8 epochs. Validation half B (never used for the weights) selects round, cutoff and epoch; test is read once.
The ablation `SimFT-flat` keeps the same structure but weights neighbours equally, so any gain over it comes from the similarity.

**Three settings. Only the SimFT arms are trained** (SimFT tau = 0.5, tau = 1.0, SimFT-flat; 5 seeds each); local-only, FedAvg and
FedAvg + fine-tune are **read from the notebook that already ran them on the same split and seeds**:

| Setting | Split | Baselines from (put the file in `results/`) |
|---|---|---|
| `louvain` | `louvain_split(n_clients=3)` | notebook 16: `amr_louvain_final_all.json` (ran on a Mac CPU: run this setting on a Mac CPU too) |
| `org10` | `organism_community(n_clients=10)` | notebook 17: `amr_org10_final_all.json` (only its FedAvg arm is needed; on Colab upload the file to Drive) |
| `ward10` | ward-mix, 10 hospitals, 10% of train/validation patients kept | notebook 18: `amr_regimes_all.json`, key `scarcity_f0.1` (ran on a Colab T4) |

Each setting has its own cell and its own results file `results/amr_simft_<setting>_<TAG>.json` (checkpointed after every seed,
resumed if present, mirrored to Drive on Colab). Run the settings in any order, on different machines if you like.

Primary metric: the **unweighted mean over hospitals** of test macro-F1, every hospital at its own validation-chosen cutoff.
**Win rule (per setting):** SimFT beats FedAvg + fine-tune, local-only and SimFT-flat, each with a paired 95% interval above zero.
Budget: rounds 8, local epochs 4, local-only epochs 32, hidden 128 (same as notebooks 16-18).

In [ ]:
# 1) Environment: Colab (clone + install + mount Drive) or local (.venv)
import os, sys
ON_COLAB = 'google.colab' in sys.modules
BRANCH = 'feat/federated-baselines'
if ON_COLAB:
    !git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
    !cd /content/Capstone-amr-fed && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
    !cd /content/Capstone-amr-fed && git log -1 --oneline
    !pip install -q torch_geometric 'flwr[simulation]' networkx scipy
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # edit to your folder if different
else:
    os.environ.setdefault('ARMD_DIR', '/Users/vikramadityasharma/Library/CloudStorage/GoogleDrive-vikramaditya.raunak@gmail.com/.shortcut-targets-by-id/1uVx5PmldPyJBMtnB9Lkl9N1rNBQrmJt-/ARMD')
import torch, ray, flwr, torch_geometric
print('colab:', ON_COLAB, '| python', sys.version.split()[0], '| torch', torch.__version__, '| flwr', flwr.__version__,
      '| ray', ray.__version__, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('ARMD_DIR:', os.environ['ARMD_DIR'])

In [ ]:
# 2) Sanity: data resolves + this is the final-protocol branch (modules below must exist)
from pathlib import Path
if ON_COLAB:
    SRC = Path('/content/Capstone-amr-fed/src')
else:
    SRC = next(p / 'src' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'amr_fed').is_dir())
sys.path.insert(0, str(SRC))
from amr_fed import config
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong - fix cell 1 and re-run.'
from amr_fed.federated import final_protocol as P, analysis as A, selection, simft, simft_runs as SR, regimes
from amr_fed.topology import personalized_weights, train_frame
print('imports OK | v3 temperatures:', P.T_GRID, '| fine-tune epochs:', P.FT_EPOCHS)

In [ ]:
# 3) Load cohort + smoke: the three partitions
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import louvain_split, organism_community, dirichlet_ward_mixture
df = load_cohort_frame()
print('rows:', f'{len(df):,}')
SEEDS = (42, 43, 44, 45, 46)
for s in SEEDS[:2]:
    a = louvain_split(df, n_clients=3, seed=s)
    print(f'louvain seed {s}: {a.nunique()} hospitals | patients: {a.value_counts().sort_values(ascending=False).tolist()}')
a = organism_community(df, n_clients=10)
print('org10 patients:', a.value_counts().sort_values(ascending=False).tolist())
w = dirichlet_ward_mixture(df, n_clients=10, alpha=0.5, seed=42)
print('ward10 seed 42 patients:', w.value_counts().sort_values(ascending=False).tolist())
print('SMOKE PASS')

In [ ]:
# 4) Shared setup + helper
import os, ray, json
os.environ.pop('AMR_RNG_PROBE', None)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed') and not any(x in m for x in ('final_protocol', 'simft_runs', 'regimes'))]:
    del _s.modules[_m]
from amr_fed.federated import final_protocol as P, analysis as A, simft_runs as SR
TAG = 'all'
RES = {}
SEARCH = [SRC.parent / 'results'] + ([Path('/content/drive/MyDrive')] if ON_COLAB else [])
COMMON = dict(seeds=SEEDS, df=df, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128)

def run_one(name, partition_fn, n_clients):
    """Run the SimFT arms of one setting (resume-safe) with the baselines read from the earlier notebook's file."""
    base, base_file = SR.baseline_from_file(name, SEARCH)
    print(f'[{name}] baselines (local-only, FedAvg, FedAvg+fine-tune) from {base_file} | seeds {base["seeds"]}')
    path = SRC.parent / 'results' / f'amr_simft_{name}_{TAG}.json'
    path.parent.mkdir(exist_ok=True)
    mirror = Path(f'/content/drive/MyDrive/amr_simft_{name}_{TAG}.json') if ON_COLAB else None
    src = path if path.exists() else (mirror if mirror and mirror.exists() else None)
    res = json.load(open(src)) if src else {}
    res['fedavg'] = base
    res = SR.run_setting(res, name, kw=dict(COMMON, partition_fn=partition_fn, n_clients=n_clients),
                         checkpoint=lambda r: P.save_json(r, path, mirror), ray_shutdown=ray.shutdown)
    P.save_json(res, path, mirror)
    RES[name] = res
    print(f'[{name}] done:', [k for k in res if not k.startswith('_')])
    return res

In [ ]:
# 5) SETTING louvain (3 hospitals) - 3 arms x 5 seeds; baselines: notebook 16
from amr_fed.partition import louvain_split
run_one('louvain', lambda d, s: louvain_split(d, n_clients=3, seed=s), 3)

In [ ]:
# 6) SETTING ward10 at 10% data - 3 arms x 5 seeds (small graphs: fast); baselines: notebook 18
from amr_fed.partition import dirichlet_ward_mixture
run_one('ward10', lambda d, s: dirichlet_ward_mixture(d, n_clients=10, alpha=0.5, seed=s), 10)

In [ ]:
# 7) SETTING org10 (10 hospitals) - 3 arms x 5 seeds, the heaviest (10x10 cross-evaluation per round): best on Colab.
#    Needs notebook 17's FedAvg arm finished (its first arm) in amr_org10_final_all.json; copy that file to results/ (or Drive).
from amr_fed.partition import organism_community
run_one('org10', lambda d, s: organism_community(d, n_clients=10, seed=s), 10)

In [ ]:
# 8) Results: louvain
print(A.simft_report(RES['louvain'])); print(); print(A.simft_weights_report(RES['louvain']))

In [ ]:
# 9) Results: ward10 at 10% data
print(A.simft_report(RES['ward10'])); print(); print(A.simft_weights_report(RES['ward10']))

In [ ]:
# 10) Results: org10
print(A.simft_report(RES['org10'])); print(); print(A.simft_weights_report(RES['org10']))

In [ ]:
# 11) Compute / provenance
import numpy as np
for name, res_ in RES.items():
    for k, v in res_.items():
        if k.startswith('_') or k == 'fedavg' or not isinstance(v, dict) or 'wall_seconds' not in v['runs'][0]:
            continue
        print(name, k, 'min per seed (mean):', round(np.mean([r['wall_seconds'] for r in v['runs']]) / 60, 1), '| gpu:', v['runs'][0]['provenance'].get('gpu'), '| git:', v['runs'][0]['provenance'].get('git'))